<img src="../Images/DSC_Logo.png" style="width: 400px;">

This notebook applies an **Automatic Speech Recognition (ASR)** workflow built on [faster-whisper](https://github.com/SYSTRAN/faster-whisper), with installation procedures based on its official GitHub repository (accessed September 25, 2025). This is one common Whisper-based setup among several: For example, [Whisper](https://github.com/openai/whisper) can also be run directly (without faster-whisper) or via more integrated pipelines such as [WhisperX](https://github.com/m-bain/whisperX).

The workflow presented in this notebook uses **Whisper for speech-to-text transcription**.

This is similar to the ASR component that runs in the background of tools like [noScribe](https://github.com/kaixxx/noScribe).

>**Using this notebook:** The workflow can be adapted throughout, but to test ASR with the default settings, only the parameters marked with **`!`** need to be changed or checked.

# 1. One-Time Setup: Install Software

The code below installs the **Python packages** needed for this notebook into the Python environment your Jupyter notebook is using. It ensures the required libraries are available so the notebook can run without import errors.

> `!` Check that you once installed the required packages in your working environment.

In [7]:
#%pip install faster-whisper imageio-ffmpeg torch "transformers[torch]>=4.23"

   ---------------------------------------- 0.0/11.7 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.7 MB 3.4 MB/s eta 0:00:04
   --- ------------------------------------ 1.0/11.7 MB 2.6 MB/s eta 0:00:05
   ----- ---------------------------------- 1.6/11.7 MB 2.6 MB/s eta 0:00:04
   ------- -------------------------------- 2.1/11.7 MB 2.6 MB/s eta 0:00:04
   -------- ------------------------------- 2.6/11.7 MB 2.6 MB/s eta 0:00:04
   ---------- ----------------------------- 3.1/11.7 MB 2.6 MB/s eta 0:00:04
   ------------- -------------------------- 3.9/11.7 MB 2.7 MB/s eta 0:00:03
   -------------- ------------------------- 4.2/11.7 MB 2.8 MB/s eta 0:00:03
   --------------- ------------------------ 4.5/11.7 MB 2.5 MB/s eta 0:00:03
   ---------------- ----------------------- 5.0/11.7 MB 2.3 MB/s eta 0:00:03
   ------------------ --------------------- 5.5/11.7 MB 2.3 MB/s eta 0:00:03
   -------------------- ------------------- 6.0/11.7 MB 2.4 MB/s eta 0:00:03
   ---

# 2. Import Packages

In [8]:
# Optional to avoid warnings:
import warnings
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", FutureWarning)

In [9]:
import os                                            # work with file/folder paths
from datetime import timedelta                       # format/handle time durations
import subprocess                                    # run external commands from Python (here: call ffmpeg)
import imageio_ffmpeg                                # provides an ffmpeg executable we can call from Python (audio conversion)
import torch                                         # lets us check whether a GPU is available
from faster_whisper import WhisperModel              # speech-to-text (ASR)
from faster_whisper import BatchedInferencePipeline  # optional: faster transcription on GPU (batching)

# path to the ffmpeg executable used by subprocess:
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

You can check the installed PyTorch version and whether your environment has access to a GPU.

Meaning of the output:
- CUDA available: False = No GPU access
- CUDA available: True = GPU access

In [10]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.8.0+cpu
CUDA available: False


# 3. Setup

## 3.1 Runtime Settings

Automatically set device and compute type depending on **hardware availability**. You don't need to know whether your hardware has a CPU or a GPU. This is **checked and selected automatically** here. 

The compute type tells the engine what kind of "number format" it should use internally while running the model. Different formats trade off speed and resource use. Batch size controls how many audio chunks are processed at once. On a GPU, a larger batch size can speed things up. On a CPU, batch size is usually kept at 1 because larger values typically don’t help.

In [11]:
device = "cuda" if torch.cuda.is_available() else "cpu"

if device == "cuda":
    compute_type = "float16"  # GPU: usually fastest
    batch_size = 16           # GPU: process several chunks at once (reduce if you get errors)
else:
    compute_type = "int8"     # CPU: usually fastest
    batch_size = 1            # CPU: process one chunk at a time

## 3.2 Select Audio File

With Python, you can easily transcribe **multiple files by looping over a list of paths** (e.g., all files in a folder) and applying the same steps to each file. In this notebook, we keep things simple and specify a single audio file. 

Below, we provide the **relative path to one audio file**. Both .wav and .mp3 files work because the transcription library uses ffmpeg under the hood to read many common audio formats. In addition, in the next step we explicitly convert the audio to a standardized format to ensure consistent processing throughout the notebook. 

To switch between example files, uncomment exactly one pair (`file_name` & `audio_file`) and keep all others commented out. To use your own audio, add the file to `../Data_Raw/`, then set `file_name` & `audio_file` in the code below to the file’s name and relative path (and comment out the other examples).

> `!` Check the selected file and optionally select a different one (comment out while uncommenting the other one)

In [14]:
#file_name = "File-A"
#audio_file = "../Data_Raw/File-A_buffy/shortened_Buffy_Seas01-Epis01.en.wav"

#file_name = "File-B"
#audio_file = "../Data_Raw/File-B_moon-landing/shortened_CA138.mp3"

#file_name = "File-C"
#audio_file = "../Data_Raw/File-C_qualitative-interview-en/shortened_JG-20170508_Interview-recording.wav"

#file_name = "File-D"
#audio_file = "../Data_Raw/File-D_Bremen-guide-low-saxon/shortened_audioguide-2025-platt-01.wav"

file_name = "File-E"
audio_file = "../Data_Raw/File-E_common-voice/common_voice_sw_30644766.mp3"   # Note: multiple files available

# 4. Preprocess Audio File

Whisper can read many audio formats and handle basic **resampling internally**. "Basic resampling" here means that Whisper can automatically adjust the sample rate of your audio file to match the model's required rate, so you do not need to manually convert it beforehand.

In this notebook, we still apply one light **preprocessing** step: we standardize the audio to a 16 kHz mono WAV.

More **advanced preprocessing** (denoising, volume normalization, echo removal, speech separation) is usually optional. It might be beneficial if you notice clear problems, such as strong background noise/echo, very uneven volume, very long silences, or heavy overlapping speech.

## 4.1 Run ffmpeg

The audio file is converted once to a 16 kHz mono WAV. This **standardized audio file** is then used for Whisper transcription to ensure consistent processing.

In [16]:
audio_16k = "../Data_Preprocessed/audio_16k_mono.wav"

# Convert with ffmpeg (standardize audio for consistent processing):
# -y                 -> overwrite output file if it already exists
# -hide_banner       -> hide ffmpeg version banner
# -loglevel error    -> show only errors (no progress/info output)
# -i <input>         -> input audio file (e.g., .mp3 or .wav)
# -ac 1              -> convert to mono (1 audio channel)
# -ar 16000          -> resample to 16,000 Hz (common format for speech models)

subprocess.run(
    [ffmpeg, "-y",
     "-hide_banner",
     "-loglevel", "error",
     "-i", audio_file,
     "-ac", "1", "-ar", "16000",
     audio_16k],
    check=True
)

print("Wrote:", audio_16k)

Wrote: ../Data_Preprocessed/audio_16k_mono.wav


# 5. Load Whisper Model

Load the **Whisper model for ASR** with the given device ("cpu" or "cuda") and precision type ("float16", "int8", etc.). You can select any [Whisper model](https://github.com/openai/whisper) size (e.g., "tiny" to "large-v3"; see section "Available models and languages" in the GitHub Repository) or provide a custom/fine-tuned model.

Here we use the Swahili fine-tuned [`adoamesh/whisper-small-swh-finetuned`](https://huggingface.co/adoamesh/whisper-small-swh-finetuned) model. Because this is a standard Hugging Face Transformers checkpoint, it is converted once to the CTranslate2 format [required by faster-whisper](https://github.com/SYSTRAN/faster-whisper#model-conversion).

In [17]:
model_id = "adoamesh/whisper-small-swh-finetuned"
ct2_model = "../Models/whisper-small-swh-finetuned-ct2"

# Convert the Hugging Face model once to the CTranslate2 format used by faster-whisper.
if not os.path.exists(os.path.join(ct2_model, "model.bin")):
    os.makedirs("../Models", exist_ok=True)
    subprocess.run(
        [
            "ct2-transformers-converter",
            "--model", model_id,
            "--output_dir", ct2_model,
            "--quantization", "float16"
        ],
        check=True
    )

model = WhisperModel(ct2_model,
                     device,
                     compute_type=compute_type)


With faster-whisper, you can run transcription "normally" or with batched inference. Batched inference is mainly a speed option for GPUs (it processes several audio chunks at once). On CPU, it usually provides little benefit, so the default (non-batched) mode is typically used. If you want the batched model (alternative):

In [ ]:
#model = BatchedInferencePipeline(model=model)

# 6. Run Transcription / ASR

The `transcribe()` function takes an audio input and produces a **transcription as a sequence of time-stamped text segments.**

You can optionally provide a **language code**. It is one of **many optional settings**. Most of the other parameters control how the decoding is done. In the setup below:

- `word_timestamps`: If enabled, the output includes estimated start/end times per word (and other results, depending on the Whisper implementation used). This increases compute cost, and word timing can be less stable for very short filler sounds or noisy speech.
- `vad_filter`: If enabled, the system tries to skip non-speech regions, which often speeds up transcription and can reduce spurious text during silence/noise. Depending on sensitivity, it may also remove very short hesitation sounds (e.g., "um", "ähm").
- `beam_size`: Influences how many alternative decoding paths are explored. Larger values can sometimes improve accuracy, but they increase runtime and the benefit varies by audio.

For details on all available parameters and their defaults, refer to the [faster-whisper](https://github.com/SYSTRAN/faster-whisper) documentation.

>Example More Optional Settings: 
>
>This example shows how you can use "hotwords" to make the transcription model pay special attention to short hesitation sounds or filler phrases in your audio.
>
>In [noScribe](https://noscribe.de/de/), **hotwords** from a separate file are passed into `transcribe()` via the `hotwords` parameter to implement the "disfluencies" on/off toggle in the noScribe application. In faster-whisper, these hotwords are inserted as extra prompt tokens before decoding, which slightly biases the decoder toward producing those words when the audio is uncertain (for example, very short filler sounds like "um" or "ähm" that can be hard to distinguish from breathing or background noise). The original OpenAI [Whisper](https://github.com/openai/whisper) implementation does not provide a `hotwords` parameter under that name, so this behavior is specific to faster-whisper. The closest equivalent in the original Whisper implementation is providing a prompt via the `initial_prompt` parameter to bias decoding in a similar direction. 
>
>If you want to test how hesitation sounds can be encouraged in the `transcribe()` call below, add a `hotwords` parameter (German example: `hotwords="Äh, das ist, es ist, ähm, nicht so einfach."`; English example: `hotwords="Uhm, okay, here's what I'm, like, thinking."`, taken from noScribe’s [prompt.yml](https://github.com/kaixxx/noScribe/blob/main/prompt.yml)). Then compare the transcription results with vs. without hotwords on an audio file that contains such fillers.

> `!` Check that no or the correct language is selected.

In [18]:
segments, info = model.transcribe(
    audio_16k,
    language="sw",   # Swahili
    word_timestamps=False, 
    vad_filter=True, 
    beam_size=5
)

segments = list(segments)  # The transcription will actually run here

`print(segments)` shows a Python list of Segment objects. In faster-whisper **transcription results**, each text **segment** (or word) includes:
- start and end time, 
- recognized text,
- the underlying token IDs (see text box below for further information),
- and several scores that can be inspected if needed. These scores are model-internal confidence signals derived from the token probabilities during decoding.

In [19]:
print(segments) # Show results

[Segment(id=1, seek=0, start=0.75, end=9.19, text='Ikumbukwe pia kuwa wajerumani wa kahifanya baga moyo kuwa makao ya mkuu ya utawala katika Afrika.', tokens=[50364, 40, 74, 2860, 2034, 826, 280, 654, 17807, 4151, 261, 1805, 260, 449, 3782, 5406, 350, 545, 351, 8791, 272, 9286, 705, 8308, 17807, 4151, 963, 25548, 2478, 275, 5279, 84, 2478, 2839, 1607, 5159, 16536, 5439, 3325, 21499, 13, 50786], avg_logprob=-0.1128042121959287, compression_ratio=1.1547619047619047, no_speech_prob=2.5769759304239415e-07, words=None, temperature=0.0)]


>Inspect Whisper Tokens:
>
>Whisper does not produce text directly. Internally, it predicts a sequence of tokens. These are numbered IDs that refer to entries in Whisper's fixed vocabulary (often whole words, parts of words, spaces, or punctuation). The final transcript is created by decoding these token IDs back into readable text.
>
>If you want to inspect the tokens in the model output more closely, you can look up what the token numbers correspond to by decoding them with the Whisper tokenizer. This lets you see the exact token sequence behind a segment. To do so, comment out the code below and insert a few tokens inside the `ids` list.

In [20]:
#import whisper
#from whisper.tokenizer import get_tokenizer
#
#tokenizer = get_tokenizer(multilingual=True, language="sw")
#
#ids = [51090, 1042, 11]                      # example: INSERT TOKEN(S)
#print([tokenizer.decode([i]) for i in ids])  # print token-by-token (roughly)

# 7. ASR Result Formatting

Since the output of faster-whisper is stored in its own custom Python object, we first convert it into a **Python data structure (a list of dictionaries)**. We define a function `to_whisper_result` that extracts only the fields we need. This makes the transcript easier to inspect, save, and process in later steps. Each segment gets a start time, an end time, the transcribed text, and (if word-level timestamps were enabled) a list of words with their own timing information.

In [21]:
def to_whisper_result(segments):
    """Convert faster-whisper segment objects into a simple list of dicts."""
    out = []
    for s in segments:
        item = {"start": float(s.start), "end": float(s.end), "text": s.text or ""}
        if getattr(s, "words", None):
            item["words"] = [
                {"word": w.word, "start": float(w.start), "end": float(w.end)}
                for w in s.words
                if w.start is not None and w.end is not None
            ]
        out.append(item)
    return out

Run conversion:

In [22]:
asr_result = to_whisper_result(segments)

In [23]:
print(asr_result) # Show results

[{'start': 0.75, 'end': 9.19, 'text': 'Ikumbukwe pia kuwa wajerumani wa kahifanya baga moyo kuwa makao ya mkuu ya utawala katika Afrika.'}]


# 8. Save Transcript

Finally, we save a **readable transcript as a text file**. For each segment, we write the transcribed text and the segment start time.

You **can adapt the format** depending on what you need for analysis in Python or external tools.

In [ ]:
# Save as ...
output_folder = "../Results/"
os.makedirs(output_folder, exist_ok=True)
txt_path = os.path.join(output_folder, f"{file_name}.txt")

# Save
with open(txt_path, "w", encoding="utf-8") as f:
    for seg in asr_result:
        start = str(timedelta(seconds=float(seg["start"])))[:-3]
        f.write(f"[{start}] {seg['text'].strip()}\n\n")

print("Saved transcript to", txt_path)